In [ ]:
!pip install --upgrade opentelemetry-api opentelemetry-sdk chromadb tree-sitter tree-sitter-python openai tqdm rank_bm25 langgraph


In [ ]:
import os
import sys
import uuid
import ast
import re
import tempfile
import subprocess
import shutil
from pathlib import Path
import chromadb
import tree_sitter
import tree_sitter_python
from openai import OpenAI
from rank_bm25 import BM25Okapi
from typing import TypedDict, List, Tuple, Optional, Dict, Any
from langgraph.graph import StateGraph, END


In [ ]:
def chunk_repo(repo_path):
    """
    Recursively scans the directory specified by repo_path for Python (.py) files,
    parses each file using Tree-sitter, and extracts top-level function definitions,
    class methods, and non-trivial class-level statements (like docstrings and properties)
    as separate chunks.
    
    Parameters:
        repo_path (str): The absolute or relative file path to the repository directory we want to scan.
        
    Returns:
        list of dict: A list of dictionaries, where each dictionary represents an extracted code chunk.
    """
    chunks = []
    
    # Conceptually, the tree-sitter-python library provides the grammar rules for the Python language,
    # and the tree-sitter core library uses those rules to initialize a Parser. The Parser reads raw code
    # and creates a structured tree representing the code's grammar (functions, classes, variables, etc.).
    try:
        language = tree_sitter.Language(tree_sitter_python.language())
        parser = tree_sitter.Parser(language)
    except Exception as error:
        print("Error initializing Tree-sitter parser:", error)
        return chunks

    # Walk the directory structure recursively to find Python source files
    for root_dir, dirs, files in os.walk(repo_path):
        for file in files:
            if file.endswith('.py'):
                full_path = os.path.join(root_dir, file)
                # Compute relative path using forward slashes for cross-platform consistency
                rel_path = os.path.relpath(full_path, repo_path).replace('\\', '/')
                
                try:
                    # Read python file contents with utf-8 encoding and replace invalid characters
                    with open(full_path, 'r', encoding='utf-8', errors='replace') as python_file:
                        content = python_file.read()
                except Exception as error:
                    print("Warning: Failed to read file", rel_path, ":", error)
                    continue

                try:
                    content_bytes = content.encode('utf-8')
                    # Parse the bytes of the file content to build the syntax tree
                    tree = parser.parse(content_bytes)
                    
                    if tree.root_node.has_error:
                        print("Warning: Syntax errors in file", rel_path, ". Skipping.")
                        continue
                    
                    # Extract top-level function and class definitions
                    for child in tree.root_node.children:
                        # 1. Handle top-level functions
                        if child.type == 'function_definition':
                            name_node = child.child_by_field_name('name')
                            name = name_node.text.decode('utf-8', errors='replace') if name_node else 'unknown'
                            
                            code_text = content_bytes[child.start_byte:child.end_byte].decode('utf-8', errors='replace')
                            start_line = child.start_point[0] + 1
                            end_line = child.end_point[0] + 1
                            
                            chunks.append({
                                "code": code_text,
                                "file_path": rel_path,
                                "class_name": "",
                                "name": name,
                                "type": "function",
                                "start_line": start_line,
                                "end_line": end_line
                            })
                        
                        # 2. Handle class definitions (split into methods and class-level properties)
                        elif child.type == 'class_definition':
                            class_name_node = child.child_by_field_name('name')
                            class_name = class_name_node.text.decode('utf-8', errors='replace') if class_name_node else 'unknown'
                            
                            # Find the class body block
                            class_block = None
                            for c in child.children:
                                if c.type == 'block':
                                    class_block = c
                                    break
                            
                            if class_block is not None:
                                for member in class_block.children:
                                    if member.type == 'function_definition':
                                        # Extract method inside class
                                        method_name_node = member.child_by_field_name('name')
                                        method_name = method_name_node.text.decode('utf-8', errors='replace') if method_name_node else 'unknown'
                                        
                                        method_code = content_bytes[member.start_byte:member.end_byte].decode('utf-8', errors='replace')
                                        m_start = member.start_point[0] + 1
                                        m_end = member.end_point[0] + 1
                                        
                                        chunks.append({
                                            "code": method_code,
                                            "file_path": rel_path,
                                            "class_name": class_name,
                                            "name": method_name,
                                            "type": "method",
                                            "start_line": m_start,
                                            "end_line": m_end
                                        })
                                    else:
                                        # Extract class-level code (docstrings, properties)
                                        member_text = content_bytes[member.start_byte:member.end_byte].decode('utf-8', errors='replace').strip()
                                        # Skip trivial items (empty, comments only, single semicolons, passes)
                                        if member_text and member_text not in (';', 'pass') and len(member_text) > 5:
                                            m_start = member.start_point[0] + 1
                                            m_end = member.end_point[0] + 1
                                            
                                            chunks.append({
                                                "code": member_text,
                                                "file_path": rel_path,
                                                "class_name": class_name,
                                                "name": class_name + "_definition",
                                                "type": "class_definition",
                                                "start_line": m_start,
                                                "end_line": m_end
                                            })
                except Exception as error:
                    print("Warning: Failed to parse file", rel_path, ":", error)
                    continue
                    
    return chunks


In [ ]:
def embed_and_store(chunks, collection_name, persist_dir, nvidia_api_key):
    """
    Converts code chunks into mathematical embeddings using NVIDIA NIM API (llama-3.2-nv-embedqa-1b-v2)
    and stores those embeddings along with their metadata inside a local, persistent Chroma vector database.
    """
    if not chunks:
        print("No chunks to embed and store.")
        return

    # Conceptually, we connect to the NVIDIA NIM API to generate embeddings in the cloud.
    # This prevents downloading heavy model weights (like PyTorch/Transformers) to your laptop.
    print("Initializing NVIDIA NIM API connection...")
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=nvidia_api_key
    )

    # Conceptually, chromadb is a database that stores these high-dimensional vectors and builds a search index
    # (using HNSW - Hierarchical Navigable Small World).
    print("Connecting to persistent Chroma database at:", persist_dir)
    client_db = chromadb.PersistentClient(path=persist_dir)

    # Get or create collection with cosine similarity configuration
    collection = client_db.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )       

    total_chunks = len(chunks)
    batch_size = 50

    from tqdm import tqdm

    print("Starting embedding and storage of", total_chunks, "chunks using NVIDIA NIM...")
    for i in tqdm(range(0, total_chunks, batch_size), desc="Embedding and storing chunks"):
        batch = chunks[i:i + batch_size]
        
        batch_codes = []
        batch_texts_to_embed = []
        for chunk in batch:
            batch_codes.append(chunk['code'])
            # Create a structured prefix text that includes the class name for method chunks
            if chunk['type'] == 'method':
                text_to_embed = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nMethod: " + chunk['name'] + "\nType: method\n\n" + chunk['code']
            elif chunk['type'] == 'class_definition':
                text_to_embed = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nType: class_definition\n\n" + chunk['code']
            else:
                text_to_embed = "File: " + chunk['file_path'] + "\nName: " + chunk['name'] + "\nType: function\n\n" + chunk['code']
            batch_texts_to_embed.append(text_to_embed)

        # Call NVIDIA NIM API to generate embeddings for the batch
        response = client.embeddings.create(
            input=batch_texts_to_embed,
            model="nvidia/llama-nemotron-embed-1b-v2",
            extra_body={"input_type": "passage", "truncate": "NONE"}
        )
        
        # Extract embeddings list from response
        batch_embeddings = []
        for data in response.data:
            batch_embeddings.append(data.embedding)

        # Generate unique IDs and extract metadata
        batch_ids = []
        for _ in batch:
            batch_ids.append(uuid.uuid4().hex)
            
        batch_metadatas = []
        for chunk in batch:
            metadata_dict = {
                "file_path": chunk["file_path"],
                "class_name": chunk["class_name"],
                "name": chunk["name"],
                "type": chunk["type"],
                "start_line": chunk["start_line"],
                "end_line": chunk["end_line"]
            }
            batch_metadatas.append(metadata_dict)

        # Add batch to Chroma DB collection
        collection.add(
            ids=batch_ids,
            embeddings=batch_embeddings,
            metadatas=batch_metadatas,
            documents=batch_codes
        )


In [ ]:
def query_repo(query, collection_name, persist_dir, nvidia_api_key, top_k=5):
    """
    Converts a natural language query into a vector embedding using NVIDIA NIM and searches the
    Chroma vector database to find the top_k most semantically similar code chunks.
    """
    # Connect to NVIDIA NIM API
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=nvidia_api_key
    )
    
    # Call NVIDIA NIM API to convert query to vector representation.
    response = client.embeddings.create(
        input=query,
        model="nvidia/llama-nemotron-embed-1b-v2",
        extra_body={"input_type": "query"}
    )
    query_embedding = response.data[0].embedding

    # Initialize Chroma client
    client_db = chromadb.PersistentClient(path=persist_dir)

    # Retrieve collection
    try:
        collection = client_db.get_collection(name=collection_name)
    except Exception as error:
        print("Error: Collection '", collection_name, "' not found:", error)
        return []

    # Query Chroma using query embedding
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    output = []
    if results is not None:
        if 'documents' in results and results['documents'] is not None:
            if len(results['documents']) > 0:
                docs = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]

                # Map results to our output format
                for idx in range(len(docs)):
                    document_text = docs[idx]
                    metadata = metadatas[idx]
                    distance = distances[idx]
                    
                    # Calculate cosine similarity (1.0 - cosine distance)
                    similarity = 1.0 - distance
                    
                    match_dict = {
                        "code": document_text,
                        "file_path": metadata["file_path"],
                        "class_name": metadata.get("class_name", ""),
                        "name": metadata["name"],
                        "type": metadata.get("type", "function"),
                        "start_line": int(metadata.get("start_line", 0)),
                        "end_line": int(metadata.get("end_line", 0)),
                        "similarity_score": similarity
                    }
                    output.append(match_dict)

    return output


In [ ]:
def tokenize_text(text):
    """
    Splits a raw text string into a list of lowercase alphanumeric word tokens.
    """
    words = []
    current_word = []
    for char in text:
        if char.isalnum():
            current_word.append(char.lower())
        else:
            if current_word:
                words.append("".join(current_word))
                current_word = []
    if current_word:
        words.append("".join(current_word))
    return words

def build_bm25_index(chunks):
    """
    Constructs two separate BM25Okapi search indexes over the extracted codebase chunks:
    one for metadata (file path, class, method/function name) and one for the code body.
    """
    metadata_corpus = []
    body_corpus = []
    for chunk in chunks:
        # Build metadata string based on chunk type
        if chunk['type'] == 'method':
            metadata_str = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name'] + "\nMethod: " + chunk['name']
        elif chunk['type'] == 'class_definition':
            metadata_str = "File: " + chunk['file_path'] + "\nClass: " + chunk['class_name']
        else:
            metadata_str = "File: " + chunk['file_path'] + "\nName: " + chunk['name']
            
        metadata_corpus.append(tokenize_text(metadata_str))
        body_corpus.append(tokenize_text(chunk['code']))
        
    metadata_index = BM25Okapi(metadata_corpus)
    body_index = BM25Okapi(body_corpus)
    return metadata_index, body_index, chunks

def bm25_search(query, metadata_index, body_index, chunks, top_k=10, metadata_weight=3.0, body_weight=1.0):
    """
    Ranks chunks using a field-weighted combination of metadata and code body BM25 scores.
    """
    tokenized_query = tokenize_text(query)
    
    # Get scores from both indexes
    metadata_scores = metadata_index.get_scores(tokenized_query)
    body_scores = body_index.get_scores(tokenized_query)
    
    scored_chunks = []
    for idx in range(len(chunks)):
        m_score = float(metadata_scores[idx])
        b_score = float(body_scores[idx])
        combined_score = (metadata_weight * m_score) + (body_weight * b_score)
        
        scored_chunks.append({
            "chunk": chunks[idx],
            "score": combined_score
        })
        
    scored_chunks.sort(key=lambda x: x["score"], reverse=True)
    
    results = []
    for item in scored_chunks[:top_k]:
        chunk = item["chunk"]
        results.append({
            "code": chunk["code"],
            "file_path": chunk["file_path"],
            "class_name": chunk.get("class_name", ""),
            "name": chunk["name"],
            "type": chunk["type"],
            "start_line": chunk["start_line"],
            "end_line": chunk["end_line"],
            "bm25_score": item["score"]
        })
    return results

def hybrid_search(query, collection_name, persist_dir, metadata_index, body_index, chunks, nvidia_api_key, top_k=5, rrf_k=60, metadata_weight=3.0, body_weight=1.0):
    """
    Fuses dense embedding similarity and field-weighted sparse BM25 keyword matching scores
    using Reciprocal Rank Fusion (RRF).
    """
    # 1. Fetch top 20 candidates from dense search
    dense_results = query_repo(query, collection_name, persist_dir, nvidia_api_key, top_k=20)
    
    # 2. Fetch top 20 candidates from BM25 search
    bm25_results = bm25_search(
        query=query,
        metadata_index=metadata_index,
        body_index=body_index,
        chunks=chunks,
        top_k=20,
        metadata_weight=metadata_weight,
        body_weight=body_weight
    )
    
    # Track rank and scores for both lists
    dense_rank = {}
    dense_scores = {}
    for idx in range(len(dense_results)):
        r = dense_results[idx]
        key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
        dense_rank[key] = idx + 1
        dense_scores[key] = r["similarity_score"]
        
    bm25_rank = {}
    bm25_scores = {}
    for idx in range(len(bm25_results)):
        r = bm25_results[idx]
        key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
        bm25_rank[key] = idx + 1
        bm25_scores[key] = r["bm25_score"]
        
    # Combine keys from both lists
    all_keys = set(dense_rank.keys()).union(set(bm25_rank.keys()))
    
    fused_results = []
    for key in all_keys:
        d_rank = dense_rank.get(key)
        rrf_dense = 1.0 / (rrf_k + d_rank) if d_rank is not None else 0.0
            
        b_rank = bm25_rank.get(key)
        rrf_bm25 = 1.0 / (rrf_k + b_rank) if b_rank is not None else 0.0
            
        rrf_score = rrf_dense + rrf_bm25
        
        # Retrieve the original chunk metadata from either of the result lists
        matched_result = None
        for r in dense_results:
            r_key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
            if r_key == key:
                matched_result = r
                break
        if matched_result is None:
            for r in bm25_results:
                r_key = r["file_path"] + "|" + r["name"] + "|" + str(r["start_line"])
                if r_key == key:
                    matched_result = r
                    break
                    
        if matched_result is not None:
            fused_results.append({
                "code": matched_result["code"],
                "file_path": matched_result["file_path"],
                "class_name": matched_result["class_name"],
                "name": matched_result["name"],
                "type": matched_result.get("type", "function"),
                "start_line": matched_result["start_line"],
                "end_line": matched_result["end_line"],
                "dense_score": dense_scores.get(key, 0.0),
                "bm25_score": bm25_scores.get(key, 0.0),
                "rrf_score": rrf_score
            })
            
    fused_results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return fused_results[:top_k]


In [ ]:
import os

# Your GitHub token and repository details
token = os.environ.get("GITHUB_TOKEN", "YOUR_GITHUB_TOKEN")
github_username = "dhruvkachhela"
repo_name = "vibesec"

# Clone the repository directly onto the Kaggle server
print("Cloning private repository...")
exit_code = os.system(f"git clone https://{token}@github.com/dhruvkachhela/vibecheck-scan")

if exit_code == 0:
    print("Success! Repository cloned successfully.")
else:
    print("Error: Failed to clone repository. Please check your username and repository name.")


In [ ]:
import os
import shutil
from kaggle_secrets import UserSecretsClient

# Generic Kaggle working folder or local working directory fallback
if os.path.exists("/kaggle/working/vibecheck-scan"):
    repo_path = "/kaggle/working/vibecheck-scan"
elif os.path.exists("./vibecheck-scan"):
    repo_path = "./vibecheck-scan"
else:
    repo_path = "."

collection_name = "repo_code_chunks"
persist_dir = "./chroma_db"

# Fetch NVIDIA API Key (Kaggle secrets or hardcoded fallback)
try:
    user_secrets = UserSecretsClient()
    nvidia_api_key = user_secrets.get_secret("nvidia_api_key")
except Exception:
    nvidia_api_key = "nvapi-YOUR_KEY_HERE"  # Set your hardcoded NVIDIA API Key string here if needed

# Automatically clean up old database folder before rebuilding to prevent duplicates
shutil.rmtree(persist_dir, ignore_errors=True)

# 1. Extract chunks
print("Step 1: Chunking codebase inside:", repo_path)
chunks = chunk_repo(repo_path)
print("Successfully extracted", len(chunks), "code chunks.")

# 2. Embed and store using NVIDIA NIM API
if chunks:
    print("\nStep 2: Embedding chunks and storing in Chroma...")
    embed_and_store(chunks, collection_name, persist_dir, nvidia_api_key)
    
    # 3. Build sparse keyword indexes
    print("\nStep 3: Compiling BM25 keyword indexes...")
    metadata_index, body_index, bm25_chunks = build_bm25_index(chunks)
    print("BM25 indexes built successfully.")
    
    print("\nIndexing completed successfully!")


In [ ]:
# Configure same collection details
collection_name = "repo_code_chunks"
persist_dir = "./chroma_db"

# Fetch NVIDIA API Key directly inside this cell
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    nvidia_api_key = user_secrets.get_secret("nvidia_api_key")
except Exception:
    nvidia_api_key = "your-nvapi-key-here"

# List of queries you want to run
queries = [
    "find the orchestrator function",
    "how does the system decide what's a false positive?",
    "which functions call the LLM?"
]

# Loop through each query and print results
for query in queries:
    print("\n" + "=" * 80)
    print(f"HYBRID QUERY: '{query}'")
    print("=" * 80)
    
    # query top_k=2 results, setting metadata_weight=3.0 and body_weight=1.0
    results = hybrid_search(
        query=query,
        collection_name=collection_name,
        persist_dir=persist_dir,
        metadata_index=metadata_index,
        body_index=body_index,
        chunks=bm25_chunks,
        nvidia_api_key=nvidia_api_key,
        top_k=2,
        rrf_k=60,
        metadata_weight=3.0,
        body_weight=1.0
    )
    
    if not results:
        print("No matches retrieved.")
    else:
        for r_idx in range(len(results)):
            result = results[r_idx]
            display_idx = r_idx + 1
            print(f"\nResult {display_idx} (RRF Score: {round(result['rrf_score'], 5)} | Dense Score: {round(result['dense_score'], 4)} | BM25 Score: {round(result['bm25_score'], 4)}):")
            print("File:", result['file_path'])
            if result['class_name']:
                print("Class:", result['class_name'], "| Name:", result['name'])
            else:
                print("Name:", result['name'])
            print("-" * 40)
            
            lines = result['code'].splitlines()
            snippet_lines = []
            for line_idx in range(min(10, len(lines))):
                snippet_lines.append(lines[line_idx])
            snippet = "\n".join(snippet_lines)
            
            if len(lines) > 10:
                snippet += "\n..."
            print(snippet)


In [ ]:
import os
import sys
import uuid
import time
import ast
import tempfile
import subprocess
import shutil
import chromadb
import tree_sitter
import tree_sitter_python
from openai import OpenAI
from rank_bm25 import BM25Okapi
from typing import TypedDict, List, Tuple, Optional, Dict, Any
from langgraph.graph import StateGraph, END

# Define ANSI color constants for terminal UI formatting
COLOR_RESET = "\033[0m"
COLOR_GREEN_BOLD = "\033[1;92m"
COLOR_GREEN = "\033[92m"
COLOR_CYAN_BOLD = "\033[1;96m"
COLOR_CYAN = "\033[96m"
COLOR_YELLOW_BOLD = "\033[1;93m"
COLOR_MAGENTA_BOLD = "\033[1;95m"

# --- Agent State Definition ---
class AgentState(TypedDict):
    question: str
    intent_category: Optional[str]
    history: List[Tuple[str, str, str]]
    current_thought: str
    action_query: Optional[str]
    final_answer: Optional[str]
    proposed_fix: Optional[Dict[str, Any]]
    sandbox_test_script: Optional[str]
    test_results: Optional[Dict[str, Any]]
    iterations: int
    verification_attempts: int
    verifier_feedback: Optional[str]
    is_grounded: bool
    node_latencies: Dict[str, List[float]]


def print_latency_benchmark_report(state: AgentState, models_info: Optional[Dict[str, str]] = None):
    """
    Computes and displays a formatted latency benchmark table summarizing execution times per node.
    """
    latencies = state.get("node_latencies", {})
    total_pipeline_time = sum(sum(durations) for durations in latencies.values())
    
    print("\n" + "=" * 90)
    print(f"{COLOR_MAGENTA_BOLD}                 DUAL-MODE AGENT LATENCY & PERFORMANCE BENCHMARK REPORT{COLOR_RESET}")
    print("=" * 90)
    print(f"{'Node Name':<20} | {'Model ID':<30} | {'Calls':<6} | {'Total (s)':<10} | {'Avg (s)':<9} | {'% Total':<7}")
    print("-" * 90)
    
    for node_name, durations in latencies.items():
        if not durations:
            continue
        call_count = len(durations)
        total_time = sum(durations)
        avg_time = total_time / call_count if call_count > 0 else 0.0
        pct = (total_time / total_pipeline_time * 100) if total_pipeline_time > 0 else 0.0
        model_name = models_info.get(node_name, "N/A") if models_info else "N/A"
        print(f"{node_name:<20} | {model_name:<30} | {call_count:<6} | {total_time:<10.3f} | {avg_time:<9.3f} | {pct:<6.1f}%")
        
    print("-" * 90)
    print(f"{COLOR_CYAN_BOLD}Total Agent Execution Latency: {total_pipeline_time:.3f} seconds{COLOR_RESET}")
    print("=" * 90 + "\n")


# --- Dual-Mode LangGraph State Machine ---
def build_agent_graph(
    collection_name: str,
    persist_dir: str,
    metadata_index: Any,
    body_index: Any,
    bm25_chunks: List[Dict[str, Any]],
    nvidia_api_key: str,
    model_classifier: str = "meta/llama-3.1-8b-instruct",
    model_reasoning: str = "z-ai/glm-5.2",
    model_verifier: str = "z-ai/glm-5.2",
    model_fix_proposal: str = "meta/llama-3.1-8b-instruct",
    model_test_generator: str = "meta/llama-3.1-8b-instruct"
):
    configured_models = {
        "intent_classifier": model_classifier,
        "reasoning": model_reasoning,
        "tool": "hybrid_search (AST+BM25+DB)",
        "verifier": model_verifier,
        "fix_proposal": model_fix_proposal,
        "execution_verifier": model_test_generator
    }

    def record_latency(state: AgentState, node_name: str, duration: float) -> Dict[str, List[float]]:
        latencies = dict(state.get("node_latencies", {}))
        if node_name not in latencies:
            latencies[node_name] = []
        latencies[node_name] = list(latencies[node_name]) + [duration]
        return latencies

    def intent_classifier_node(state: AgentState):
        start_time = time.time()
        question = state["question"]
        
        system_prompt = (
            "You are a specialized Intent Classifier for an enterprise Codebase RAG system.\n"
            "Analyze the user request and classify it into EXACTLY one category:\n\n"
            "1. QA: The user is asking an informational, architectural, or explanatory question about how the codebase operates.\n"
            "2. FIX_PROPOSAL: The user is reporting a bug, asking to fix an issue, patch code, or resolve an exception.\n\n"
            "Respond ONLY with a single word: QA or FIX_PROPOSAL."
        )
        
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        response = client.chat.completions.create(
            model=model_classifier,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"User Request: {question}"}
            ],
            temperature=0.0,
            max_tokens=20
        )
        
        intent = response.choices[0].message.content.strip().upper()
        category = "FIX_PROPOSAL" if "FIX" in intent else "QA"
        
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "intent_classifier", elapsed_time)
        
        print(f"\n--- [{COLOR_CYAN_BOLD}INTENT CLASSIFIER{COLOR_RESET}] (Model: {model_classifier} | Time: {elapsed_time:.3f}s) ---")
        print(f"Query: '{question}' -> Category: {category}")
        
        return {
            "intent_category": category,
            "node_latencies": updated_latencies
        }

    def reasoning_node(state: AgentState):
        start_time = time.time()
        iterations = state.get("iterations", 0) + 1
        history_list = state.get("history", [])
        verifier_feedback = state.get("verifier_feedback", None)
        intent = state.get("intent_category", "QA")
        
        history_str = ""
        for i, (thought, action, obs) in enumerate(history_list):
            history_str += f"\nTurn {i+1}:\nThought: {thought}\nAction: search(\"{action}\")\nObservation:\n{obs}\n"
            
        system_prompt = (
            "You are a Senior Principal Software Architect navigating a complex codebase.\n"
            "Your objective is to gather necessary code context using precise search queries.\n\n"
            "To search the codebase, output EXACTLY:\n"
            "Thought: [your reasoning for what specific symbol, file, or class to search]\n"
            "Action: search(\"[exact search terms or function names]\")\n\n"
            "When you have retrieved sufficient code context, output EXACTLY:\n"
            "Thought: [your conclusion that sufficient context has been gathered]\n"
            "Final Answer: [your response]\n\n"
            "CRITICAL RESPONSE GUIDELINES FOR FINAL ANSWER:\n"
            "1. For QA queries: Output a clear, structured, high-level natural language explanation in plain English. "
            "Explain the architecture, components, and frameworks clearly. DO NOT dump raw function source code bodies.\n"
            "2. Cite relevant file paths (e.g., scanner/indexer.py) and function names to ground your explanation.\n"
        )
        
        if verifier_feedback and state.get("verification_attempts", 0) > 0:
            system_prompt += f"\n\nCRITICAL ATTENTION - PREVIOUS REJECTION FEEDBACK FROM VERIFIER:\n{verifier_feedback}\n"
            
        user_prompt = f"Question: {state['question']}\n\nSearch History:\n{history_str if history_str else 'No searches conducted yet.'}\n\nDetermine next step."
        
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        response = client.chat.completions.create(
            model=model_reasoning,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=2048
        )
        
        text = response.choices[0].message.content.strip()
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "reasoning", elapsed_time)
        
        print(f"\n--- [{COLOR_CYAN_BOLD}REASONING AGENT{COLOR_RESET}] Iteration {iterations} (Model: {model_reasoning} | Time: {elapsed_time:.3f}s) ---")
        print(text)
        
        current_thought = ""
        action_query = None
        final_answer = None
        
        lines = text.split("\n")
        for line in lines:
            if line.startswith("Thought:"):
                current_thought = line.replace("Thought:", "").strip()
            elif line.startswith("Action:"):
                act = line.replace("Action:", "").strip()
                if "search(" in act and act.endswith(")"):
                    action_query = act.split("search(")[1][:-1].strip("\"'")
            elif line.startswith("Final Answer:"):
                final_answer = text.split("Final Answer:")[1].strip()
                
        if not action_query and not final_answer:
            current_thought = text
            final_answer = text
            
        return {
            "current_thought": current_thought,
            "action_query": action_query,
            "final_answer": final_answer,
            "iterations": iterations,
            "node_latencies": updated_latencies
        }

    def tool_node(state: AgentState):
        start_time = time.time()
        action = state["action_query"]
        
        results = hybrid_search(
            query=action,
            collection_name=collection_name,
            persist_dir=persist_dir,
            metadata_index=metadata_index,
            body_index=body_index,
            chunks=bm25_chunks,
            nvidia_api_key=nvidia_api_key,
            top_k=5,
            metadata_weight=3.0,
            body_weight=1.0
        )
        
        observation = ""
        for r in results:
            observation += f"File: {r['file_path']} | Name: {r['name']} | Type: {r['type']}\n"
            observation += f"Code:\n{r['code']}\n"
            observation += "-" * 40 + "\n"
            
        if not results:
            observation = "No matching code chunks found."
            
        new_history = list(state["history"])
        new_history.append((state["current_thought"], action, observation))
        
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "tool", elapsed_time)
        
        print(f"\n--- [{COLOR_CYAN}TOOL CALL{COLOR_RESET}] (Time: {elapsed_time:.3f}s) ---")
        print(f"Query: '{action}' -> Retrieved {len(results)} chunks.")
        
        return {
            "history": new_history,
            "action_query": None,
            "node_latencies": updated_latencies
        }

    def verifier_node(state: AgentState):
        start_time = time.time()
        final_answer = state["final_answer"]
        history_list = state.get("history", [])
        attempts = state.get("verification_attempts", 0) + 1
        
        all_observations = ""
        for step_idx, (thought, action, observation) in enumerate(history_list):
            all_observations += f"\n--- Step {step_idx + 1} (Query: {action}) ---\n{observation}\n"
            
        if not all_observations.strip():
            all_observations = "No code chunks were retrieved during search."
            
        verifier_system_prompt = (
            "You are a strict Grounding Critic and Anti-Hallucination Audit Engine.\n"
            "Your sole objective is to audit proposed QA answers against raw retrieved code observations.\n\n"
            "AUDIT RULES:\n"
            "1. Every component, framework, file reference, and architectural claim in the Proposed Answer MUST be explicitly supported by the retrieved code observations.\n"
            "2. If all claims are grounded in code, output:\n"
            "VERDICT: SUPPORTED\n\n"
            "3. If any claim is ungrounded, fabricated, or inaccurate, output:\n"
            "VERDICT: UNSUPPORTED\n"
            "Unsupported Claims:\n"
            "- [Detail each ungrounded claim]\n"
        )
        
        verifier_user_prompt = f"Question: {state['question']}\n\nRetrieved Code Observations:\n{all_observations}\n\nProposed Answer:\n{final_answer}"
        
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        response = client.chat.completions.create(
            model=model_verifier,
            messages=[
                {"role": "system", "content": verifier_system_prompt},
                {"role": "user", "content": verifier_user_prompt}
            ],
            temperature=0.0,
            max_tokens=2048
        )
        
        verifier_text = response.choices[0].message.content.strip()
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "verifier", elapsed_time)
        
        print(f"\n--- [{COLOR_YELLOW_BOLD}GROUNDING VERIFIER CRITIC{COLOR_RESET}] Attempt {attempts} (Model: {model_verifier} | Time: {elapsed_time:.3f}s) ---")
        print(verifier_text)
        
        is_grounded = "VERDICT: SUPPORT" in verifier_text.upper() and "UNSUPPORT" not in verifier_text.upper()
        
        if is_grounded or attempts >= 3:
            updated_answer = final_answer if final_answer else ""
            if not is_grounded:
                updated_answer += f"\n\n[WARNING: Partially Grounded]\nVerifier Feedback:\n{verifier_text}"
            return {
                "is_grounded": is_grounded,
                "verifier_feedback": verifier_text,
                "verification_attempts": attempts,
                "final_answer": updated_answer,
                "node_latencies": updated_latencies
            }
        else:
            return {
                "is_grounded": False,
                "verifier_feedback": verifier_text,
                "verification_attempts": attempts,
                "final_answer": None,
                "node_latencies": updated_latencies
            }

    def fix_proposal_node(state: AgentState):
        start_time = time.time()
        question = state["question"]
        history_list = state.get("history", [])
        verifier_feedback = state.get("verifier_feedback", None)
        
        all_observations = ""
        for step_idx, (thought, action, observation) in enumerate(history_list):
            all_observations += f"\n--- Step {step_idx + 1} (Query: {action}) ---\n{observation}\n"
            
        system_prompt = (
            "You are a Staff Security Engineer and Automated Patch Specialist.\n"
            "Analyze the bug report and retrieved codebase observations to synthesize a production-grade code fix proposal.\n\n"
            "You MUST output your fix proposal using EXACTLY the following key-value format:\n\n"
            "ROOT_CAUSE: [Provide a precise 2-3 sentence technical diagnosis of why the existing code fails]\n\n"
            "FILE_PATH: [Provide the exact relative file path of the target file to modify]\n\n"
            "ORIGINAL_CODE:\n"
            "```python\n"
            "[exact original lines to be replaced from the observation]\n"
            "```\n\n"
            "REPLACEMENT_CODE:\n"
            "```python\n"
            "[exact replacement code fixing the bug cleanly without syntax errors]\n"
            "```\n\n"
            "EXPLANATION: [Step-by-step explanation of how the patch resolves the bug safely]\n"
        )
        
        if verifier_feedback:
            system_prompt += f"\n\nCRITICAL PREVIOUS TEST FAILURE FEEDBACK:\n{verifier_feedback}\n"
            
        user_prompt = f"Bug Report: {question}\n\nRetrieved Code Observations:\n{all_observations}\n\nSynthesize structured fix proposal now."
        
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        response = client.chat.completions.create(
            model=model_fix_proposal,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1,
            max_tokens=3072
        )
        
        text = response.choices[0].message.content.strip()
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "fix_proposal", elapsed_time)
        
        print(f"\n--- [{COLOR_CYAN_BOLD}FIX PROPOSAL AGENT{COLOR_RESET}] (Model: {model_fix_proposal} | Time: {elapsed_time:.3f}s) ---")
        print(text)
        
        file_path = "target_file.py"
        root_cause = "Bug detected in target method"
        original_code = ""
        replacement_code = ""
        explanation = ""
        
        if "FILE_PATH:" in text:
            file_path = text.split("FILE_PATH:")[1].split("\n")[0].strip()
        if "ROOT_CAUSE:" in text:
            root_cause = text.split("ROOT_CAUSE:")[1].split("FILE_PATH:")[0].strip() if "FILE_PATH:" in text else text.split("ROOT_CAUSE:")[1].split("\n")[0].strip()
        if "ORIGINAL_CODE:" in text and "REPLACEMENT_CODE:" in text:
            orig_part = text.split("ORIGINAL_CODE:")[1].split("REPLACEMENT_CODE:")[0]
            if "```python" in orig_part:
                original_code = orig_part.split("```python")[1].split("```")[0].strip()
            elif "```" in orig_part:
                original_code = orig_part.split("```")[1].split("```")[0].strip()
            else:
                original_code = orig_part.strip()
                
            repl_part = text.split("REPLACEMENT_CODE:")[1]
            if "EXPLANATION:" in repl_part:
                repl_code_part = repl_part.split("EXPLANATION:")[0]
                explanation = repl_part.split("EXPLANATION:")[1].strip()
            else:
                repl_code_part = repl_part
                
            if "```python" in repl_code_part:
                replacement_code = repl_code_part.split("```python")[1].split("```")[0].strip()
            elif "```" in repl_code_part:
                replacement_code = repl_code_part.split("```")[1].split("```")[0].strip()
            else:
                replacement_code = repl_code_part.strip()
                
        proposed_fix = {
            "file_path": file_path,
            "root_cause": root_cause,
            "original_code": original_code,
            "replacement_code": replacement_code,
            "explanation": explanation,
            "raw_text": text
        }
        
        return {
            "proposed_fix": proposed_fix,
            "node_latencies": updated_latencies
        }

    def execution_verifier_node(state: AgentState):
        start_time = time.time()
        proposed_fix = state.get("proposed_fix", {})
        attempts = state.get("verification_attempts", 0) + 1
        replacement_code = proposed_fix.get("replacement_code", "")
        original_code = proposed_fix.get("original_code", "")
        file_path = proposed_fix.get("file_path", "module.py")
        
        print(f"\n--- [{COLOR_YELLOW_BOLD}EXECUTION VERIFIER AGENT{COLOR_RESET}] Attempt {attempts} (Model: {model_test_generator}) ---")
        
        syntax_valid = False
        syntax_error_msg = ""
        try:
            ast.parse(replacement_code)
            syntax_valid = True
            print("✓ AST Syntax Check: Passed (Valid Python Syntax)")
        except SyntaxError as se:
            syntax_error_msg = f"SyntaxError in replacement code: {se}"
            print(f"✗ AST Syntax Check: Failed -> {syntax_error_msg}")
            
        if not syntax_valid:
            elapsed_time = time.time() - start_time
            updated_latencies = record_latency(state, "execution_verifier", elapsed_time)
            if attempts >= 3:
                final_report = (
                    f"### Category 2 Bug Fix Proposal (AST Error)\n\n"
                    f"**Target File**: `{file_path}`\n\n"
                    f"**Root Cause**: {proposed_fix.get('root_cause', '')}\n\n"
                    f"**Proposed Replacement Code**:\n```python\n{replacement_code}\n```\n\n"
                    f"[WARNING: Syntax Validation Failed: {syntax_error_msg}]"
                )
                return {
                    "verification_attempts": attempts,
                    "test_results": {"passed": False, "output": syntax_error_msg},
                    "final_answer": final_report,
                    "node_latencies": updated_latencies
                }
            else:
                return {
                    "verification_attempts": attempts,
                    "verifier_feedback": f"Syntax Error in replacement code: {syntax_error_msg}. Correct syntax in replacement_code.",
                    "test_results": {"passed": False, "output": syntax_error_msg},
                    "node_latencies": updated_latencies
                }
                
        test_gen_prompt = (
            "You are a Senior QA Test Engineer creating an automated unit test script.\n"
            "Given a bug report and a proposed fix, generate a self-contained Python test script (using pytest or unittest)\n"
            "that reproduces the bug and asserts that the replacement fix operates correctly.\n\n"
            "Output ONLY valid Python code inside a single ```python codeblock.\n"
        )
        
        user_test_prompt = (
            f"Bug Report: {state['question']}\n"
            f"Target File: {file_path}\n"
            f"Proposed Replacement Code:\n{replacement_code}\n"
            "Generate unit test script now."
        )
        
        client = OpenAI(
            base_url="https://integrate.api.nvidia.com/v1",
            api_key=nvidia_api_key
        )
        
        test_response = client.chat.completions.create(
            model=model_test_generator,
            messages=[
                {"role": "system", "content": test_gen_prompt},
                {"role": "user", "content": user_test_prompt}
            ],
            temperature=0.0,
            max_tokens=2048
        )
        
        test_script_text = test_response.choices[0].message.content.strip()
        if "```python" in test_script_text:
            test_code = test_script_text.split("```python")[1].split("```")[0].strip()
        elif "```" in test_script_text:
            test_code = test_script_text.split("```")[1].split("```")[0].strip()
        else:
            test_code = test_script_text.strip()
            
        print("✓ Dynamic Test Case Generator: Script Generated.")
        
        temp_dir = tempfile.mkdtemp(prefix="agent_sandbox_")
        test_passed = False
        exec_output = ""
        
        try:
            test_file_path = os.path.join(temp_dir, "test_bug_fix.py")
            with open(test_file_path, "w", encoding="utf-8") as tf:
                tf.write(test_code)
                
            res = subprocess.run(
                [sys.executable, "-m", "pytest", test_file_path],
                capture_output=True,
                text=True,
                timeout=15
            )
            
            exec_output = res.stdout + "\n" + res.stderr
            if res.returncode == 0 or "passed" in res.stdout.lower():
                test_passed = True
                print("✓ Sandboxed Test Execution Engine: PASSED (Bug Resolution Confirmed)")
            else:
                print(f"! Sandboxed Test Execution Engine: FAILED (Exit Code {res.returncode})")
        except Exception as e:
            exec_output = f"Execution error: {str(e)}"
            print(f"! Sandbox Execution Exception: {str(e)}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
            
        elapsed_time = time.time() - start_time
        updated_latencies = record_latency(state, "execution_verifier", elapsed_time)
        
        final_patch_report = (
            f"### Category 2 Bug Fix Proposal (Verified)\n\n"
            f"**Target File**: `{file_path}`\n\n"
            f"**Root Cause Diagnosis**:\n{proposed_fix.get('root_cause', 'N/A')}\n\n"
            f"**Original Code**:\n```python\n{original_code}\n```\n\n"
            f"**Proposed Replacement Code**:\n```python\n{replacement_code}\n```\n\n"
            f"**Explanation**:\n{proposed_fix.get('explanation', 'Bug fix applied cleanly.')}\n\n"
            f"--- \n### Dynamic Execution Verification\n"
            f"- **AST Syntax Status**: Passed\n"
            f"- **Sandbox Unit Test Status**: {'PASSED (Confirmed Bug Resolved)' if test_passed else 'FAILED'}\n"
            f"```text\n{exec_output[:500] if exec_output else 'Executed successfully.'}\n```"
        )
        
        if test_passed or attempts >= 3:
            return {
                "verification_attempts": attempts,
                "sandbox_test_script": test_code,
                "test_results": {"passed": test_passed, "output": exec_output},
                "final_answer": final_patch_report,
                "node_latencies": updated_latencies
            }
        else:
            return {
                "verification_attempts": attempts,
                "verifier_feedback": f"Sandbox Test Execution Failed:\n{exec_output[:400]}\nAdjust the replacement_code to fix this failure.",
                "test_results": {"passed": False, "output": exec_output},
                "node_latencies": updated_latencies
            }

    # Graph Routers
    def route_reasoning(state: AgentState):
        if state.get("final_answer") is not None or state.get("iterations", 0) >= 15:
            if state.get("intent_category") == "FIX_PROPOSAL":
                return "fix_proposal"
            else:
                return "verifier"
        return "tool"

    def route_qa_verification(state: AgentState):
        if state.get("final_answer") is not None:
            return "end"
        return "re_reason"

    def route_fix_verification(state: AgentState):
        if state.get("final_answer") is not None:
            return "end"
        return "re_fix"

    # Assemble Workflow
    workflow = StateGraph(AgentState)
    workflow.add_node("intent_classifier", intent_classifier_node)
    workflow.add_node("reasoning", reasoning_node)
    workflow.add_node("tool", tool_node)
    workflow.add_node("verifier", verifier_node)
    workflow.add_node("fix_proposal", fix_proposal_node)
    workflow.add_node("execution_verifier", execution_verifier_node)
    
    workflow.set_entry_point("intent_classifier")
    workflow.add_edge("intent_classifier", "reasoning")

    workflow.add_conditional_edges(
        "reasoning",
        route_reasoning,
        {
            "verifier": "verifier",
            "fix_proposal": "fix_proposal",
            "tool": "tool"
        }
    )
    
    workflow.add_conditional_edges(
        "verifier",
        route_qa_verification,
        {
            "end": END,
            "re_reason": "reasoning"
        }
    )

    workflow.add_conditional_edges(
        "execution_verifier",
        route_fix_verification,
        {
            "end": END,
            "re_fix": "fix_proposal"
        }
    )

    workflow.add_edge("tool", "reasoning")
    workflow.add_edge("fix_proposal", "execution_verifier")
    
    app = workflow.compile()
    app.configured_models = configured_models
    return app

print("Compiling Optimal Hybrid Dual-Mode Agent (Classifier/Fix: Llama-3.1-8B | Reasoning/Verifier: GLM-5.2)...")
agent_app = build_agent_graph(
    collection_name=collection_name,
    persist_dir=persist_dir,
    metadata_index=metadata_index,
    body_index=body_index,
    bm25_chunks=bm25_chunks,
    nvidia_api_key=nvidia_api_key
)
print("Dual-Mode Agent graph compiled successfully.")


In [ ]:
test_queries = [
    "what LLM and it's framework are we using in it?",
    "In dataflow.py inside def execute_tool function, there is a bug with sanitization where it marks sanitization = False for all vulnerabilities. Propose a fix for this bug and verify it."
]

print("Testing Dual-Mode Agent Pipeline:\n")

for query in test_queries:
    print("=" * 90)
    print(f"USER QUERY: '{query}'")
    print("=" * 90)
    
    initial_state = {
        "question": query,
        "intent_category": None,
        "history": [],
        "current_thought": "",
        "action_query": None,
        "final_answer": None,
        "proposed_fix": None,
        "sandbox_test_script": None,
        "test_results": None,
        "iterations": 0,
        "verification_attempts": 0,
        "verifier_feedback": None,
        "is_grounded": False,
        "node_latencies": {}
    }
    
    final_state = agent_app.invoke(initial_state)
    
    print("\n" + "-" * 90)
    print(f"{COLOR_GREEN_BOLD}FINAL VERIFIED AGENT OUTPUT:{COLOR_RESET}")
    print(f"{COLOR_GREEN}{final_state.get('final_answer')}{COLOR_RESET}")
    print("-" * 90)
    
    # Print colored latency benchmark table
    print_latency_benchmark_report(final_state, agent_app.configured_models)
